In [ ]:
from pathlib import Path

# Find the repository root so these notebooks work on other computers.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "Dataset Specs").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "Dataset Specs").is_dir():
    raise FileNotFoundError("Start Jupyter from inside the Hospital Dataset project folder.")

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

data_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/80_20"
)

arima_df = pd.read_csv(
    data_dir / "arima_occupancy_predictions.csv",
    parse_dates=["date"]
)

rf_df = pd.read_csv(
    data_dir / "random_forest_occupancy_predictions.csv",
    parse_dates=["date"]
)

gb_df = pd.read_csv(
    data_dir / "gradient_boosting_occupancy_predictions.csv",
    parse_dates=["date"]
)

df_fusion = arima_df.merge(
    rf_df, on="date", suffixes=("_arima", "_rf")
)

df_fusion = df_fusion.merge(
    gb_df, on="date"
)

print(df_fusion.head())
print("Combined rows:", len(df_fusion))
print("Columns:", df_fusion.columns.tolist())

In [ ]:
# Check that all three files contain the same actual occupancy
assert (df_fusion["actual_occupied_beds_x"] ==
        df_fusion["actual_occupied_beds_y"]).all(), \
    "Actual occupancy values do not match!"

# Keep one actual occupancy column
df_fusion = df_fusion.rename(
    columns={"actual_occupied_beds_x": "actual_occupied_beds"}
)

df_fusion = df_fusion.drop(
    columns=["actual_occupied_beds_y"]
)

print(df_fusion.columns.tolist())
print("Rows:", len(df_fusion))
print("Missing values:", df_fusion.isna().sum().sum())

In [ ]:
# Average the five Random Forest predictions
df_fusion["rf_predicted_occupied_beds"] = df_fusion[
    ["rf_1", "rf_2", "rf_3", "rf_4", "rf_5"]
].mean(axis=1)

# Average the five Gradient Boosting predictions
df_fusion["gb_predicted_occupied_beds"] = df_fusion[
    ["gb_1", "gb_2", "gb_3", "gb_4", "gb_5"]
].mean(axis=1)

print(df_fusion[
    [
        "date",
        "actual_occupied_beds",
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].head())

In [ ]:
# Simple average fusion of the three models
df_fusion["fusion_predicted_occupied_beds"] = df_fusion[
    [
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds"
    ]
].mean(axis=1)

print(df_fusion[
    [
        "date",
        "actual_occupied_beds",
        "arima_predicted_occupied_beds",
        "rf_predicted_occupied_beds",
        "gb_predicted_occupied_beds",
        "fusion_predicted_occupied_beds"
    ]
].head())

In [ ]:
models = {
    "ARIMA": "arima_predicted_occupied_beds",
    "Random Forest": "rf_predicted_occupied_beds",
    "Gradient Boosting": "gb_predicted_occupied_beds",
    "Fusion Average": "fusion_predicted_occupied_beds"
}

results = []

actual = df_fusion["actual_occupied_beds"]

for model_name, prediction_col in models.items():
    predicted = df_fusion[prediction_col]

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    results.append({
        "Model": model_name,
        "RMSE (beds)": rmse,
        "MAE (beds)": mae,
        "MAPE (%)": mape
    })

results_df = pd.DataFrame(results).sort_values("RMSE (beds)")

print(results_df.round(3).to_string(index=False))

In [ ]:
from pathlib import Path

output_dir = Path(
    f"{PROJECT_ROOT}/"
    "synthetic_hospital_outputs/80_20"
)
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_dir / "fusion_baseline_metrics.csv",
    index=False
)

df_fusion.to_csv(
    output_dir / "fusion_baseline_predictions.csv",
    index=False
)

print("Baseline fusion metrics and predictions saved.")
print("Folder:", output_dir)